# KMMMU evaluation tutorial

This notebook runs a simple evaluation loop:

1. Load **HAERAE-HUB/KMMMU** from Hugging Face.
2. Build multimodal prompts (text plus images).
3. Run offline inference with **vLLM** using `LLM.generate`.
4. Parse the model output with **Math Verify** `parse`.
5. Send gold and prediction to an LLM judge using **LiteLLM**.
6. Compute accuracy.

Helper code (prompts and small utilities) is kept in a separate Python file: `kmmmu_utils.py`.


## 1. Config and dataset load

Edit the model names if needed.  
This cell loads the full dataset (no `head()` slicing).


In [ ]:
DATASET_NAME = "HAERAE-HUB/KMMMU"
DATA_FILES = "kmmmu.csv"   # per dataset card
SPLIT = "train"
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
MAX_NEW_TOKENS = 8192
TEMPERATURE = 0.6

# LLM judge through LiteLLM (set the provider API key in your environment, e.g. OPENAI_API_KEY)
JUDGE_MODEL = "gpt-5-mini"
OUT_CSV = "kmmmu_eval.csv"

import ast
import pandas as pd
from datasets import load_dataset

df = load_dataset("HAERAE-HUB/KMMMU", data_files="kmmmu.csv")["train"].to_pandas()

questions = df['question'].values
question_types = df['question_type'].values
golds = df['answer'].values

image_urls_list = [eval(s) for s in df['image_link']]
max_images_in_split = max(len(x) for x in image_urls_list)

## 2. Helper utilities

We import:
- `pil_to_base64` for image encoding
- `fetch_image` for loading images from URLs
- the system prompts used for generation and judging


In [ ]:
from kmmmu_utils import pil_to_base64, fetch_image, SYSTEM_PROMPT, JUDGE_SYSTEM_PROMPT

## 3. vLLM setup

This initializes:
- a tokenizer to apply the chat template
- the vLLM engine
- sampling parameters


In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    limit_mm_per_prompt={"image": max_images_in_split},
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS,
)

## 4. Build prompts

For each example:
- start a chat message list
- add all images as `image_url` entries (data URI with base64)
- apply the chat template
- collect prompts in `llm_inputs`


In [ ]:
llm_inputs = []
for q, urls in zip(questions, image_urls_list):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "text", "text": q}
        ]},
    ]


    for u in urls:
        img = fetch_image(u)
        img_b64 = pil_to_base64(img)
        messages[1]['content'].append(
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
        )
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    llm_inputs.append(prompt)

## 5. Run generation

Single `llm.generate(...)` call over the full prompt list.


In [ ]:
outputs = llm.generate(llm_inputs, sampling_params=sampling_params)
preds = [o.outputs[0].text for o in outputs]

## 6. Parse predictions with Math Verify

We parse the raw text outputs into a normalized representation.


In [ ]:
from math_verify import parse

parsed_preds = [parse(p) for p in preds]
parsed_preds_str = [str(x) for x in parsed_preds]

preds[0], parsed_preds_str[0]

## 7. LLM judge with LiteLLM

This builds one message per example and calls `batch_completion` once.

Important: set your provider API key in the environment before running, for example `OPENAI_API_KEY`.


In [ ]:
from litellm import batch_completion
import os
judge_messages = []
for q, qt, gold, pred, parsed_pred in zip(
    questions, question_types, golds, preds, parsed_preds_str
):
    judge_user_prompt = (
        f"Question:\n{q}\n\n"
        f"Gold answer:\n{gold}\n\n"
        f"Model output:\n{pred}\n\n"
        f"Parsed model output (Math-Verify parse):\n{parsed_pred}\n\n"
        "Is the model answer correct? Reply only with true or false."
    )
    judge_messages.append(
        [
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": judge_user_prompt},
        ]
    )

# Single call: pass the entire list at once
judge_responses = batch_completion(
    model=JUDGE_MODEL,
    messages=judge_messages,
    temperature=1.0,
)

judge_texts = [
    r["choices"][0]["message"]["content"].strip().lower() for r in judge_responses
]

correct = [t.startswith("true") for t in judge_texts]
num_correct = sum(correct)
accuracy = num_correct / len(correct)

accuracy, num_correct, len(correct)

## 8. Inspect one judge response (optional)

This is useful for debugging the judge output format.


In [ ]:
judge_responses[0]

## 9. Next steps

You can save a CSV by creating a DataFrame from:
- `questions`
- `golds`
- `preds`
- `parsed_preds_str`
- `judge_texts`
- `correct`

If you already have a saving cell elsewhere, you can ignore this section.
